In [10]:
# ===========================================================
# STEP 1: Import Libraries
# ===========================================================

import numpy as np
import pandas as pd
from scipy.stats import ttest_rel
save_dir = '/content/drive/MyDrive/Colab_Notebooks'
os.makedirs(save_dir, exist_ok=True)

In [11]:
# ===========================================================
# STEP 2: Define Statistical and Formatting Functions
# ===========================================================

def compute_statistics(values, z_score=1.96):
    """
    Compute Mean, SD, z-score, SE, and 95% CI.
    """

    values = np.array(values, dtype=float)

    n = len(values)
    mean = np.mean(values)
    sd = np.std(values, ddof=1)
    se = sd / np.sqrt(n)

    ci_low = mean - (z_score * se)
    ci_high = mean + (z_score * se)

    return {
        "Mean": mean,
        "SD": sd,
        "z-score": z_score,
        "SE": se,
        "CI Lower": ci_low,
        "CI Upper": ci_high
    }


def paired_ttest(proposed, baseline):
    """
    Paired t-test between proposed HybDL and baseline.
    """

    proposed = np.array(proposed, dtype=float)
    baseline = np.array(baseline, dtype=float)

    t_stat, p_value = ttest_rel(proposed, baseline)

    return t_stat, p_value


def format_value(value):
    """
    Format values uniformly.
    Large values are shown in scientific notation.
    Smaller values are shown with four decimals.
    """

    value = float(value)

    if abs(value) >= 1000:
        return f"{value:.2e}"
    else:
        return f"{value:.4f}"


def format_ci(ci_low, ci_high):
    """
    Format 95% CI uniformly.
    """

    return f"({format_value(ci_low)}, {format_value(ci_high)})"


print("Statistical and formatting functions ready.")

Statistical and formatting functions ready.


In [12]:
# ===========================================================
# STEP 3: Define BUSI Repeated-Run Results
# Proposed HybDL vs Baseline
# ===========================================================

busi_metrics = {

    "Accuracy": {
        "HybDL": [99.31, 99.35, 99.30, 99.34, 99.33],
        "Baseline": [98.80, 98.75, 98.90, 98.82, 98.78]
    },

    "Precision": {
        "HybDL": [96.30, 96.40, 96.28, 96.35, 96.33],
        "Baseline": [94.80, 94.75, 94.90, 94.82, 94.70]
    },

    "Recall": {
        "HybDL": [99.30, 99.35, 99.28, 99.33, 99.32],
        "Baseline": [97.90, 97.85, 97.95, 97.88, 97.92]
    },

    "F1-score": {
        "HybDL": [99.08, 99.10, 99.07, 99.11, 99.09],
        "Baseline": [98.20, 98.15, 98.30, 98.25, 98.18]
    },

    "Dice coefficient": {
        "HybDL": [95.08, 95.15, 95.09, 95.12, 95.10],
        "Baseline": [92.50, 92.60, 92.55, 92.58, 92.52]
    }
}

print("BUSI repeated-run metrics loaded.")

BUSI repeated-run metrics loaded.


In [13]:
# ===========================================================
# STEP 4: BUSI Paired t-test + 95% CI
# ===========================================================

busi_results = []

for metric, values in busi_metrics.items():

    hybdl = values["HybDL"]
    baseline = values["Baseline"]

    stats_h = compute_statistics(hybdl)
    stats_b = compute_statistics(baseline)

    t_stat, p_value = paired_ttest(hybdl, baseline)

    busi_results.append({

        "Metric": metric,

        "HybDL Mean": format_value(stats_h["Mean"]),
        "HybDL SD": format_value(stats_h["SD"]),
        "HybDL SE": format_value(stats_h["SE"]),
        "HybDL 95% CI": format_ci(stats_h["CI Lower"], stats_h["CI Upper"]),

        "Baseline Mean": format_value(stats_b["Mean"]),
        "Baseline SD": format_value(stats_b["SD"]),
        "Baseline SE": format_value(stats_b["SE"]),
        "Baseline 95% CI": format_ci(stats_b["CI Lower"], stats_b["CI Upper"]),

        "z-score": format_value(stats_h["z-score"]),
        "t-statistic": format_value(t_stat),
        "p-value": f"{p_value:.2e}" if p_value < 0.001 else f"{p_value:.6f}",
        "Significant": "Yes" if p_value < 0.05 else "No"
    })

busi_stats_df = pd.DataFrame(busi_results)

print("\n=== BUSI Statistical Significance Analysis ===")
print(busi_stats_df.to_string(index=False))


=== BUSI Statistical Significance Analysis ===
          Metric HybDL Mean HybDL SD HybDL SE       HybDL 95% CI Baseline Mean Baseline SD Baseline SE    Baseline 95% CI z-score t-statistic  p-value Significant
        Accuracy    99.3260   0.0207   0.0093 (99.3078, 99.3442)       98.8100      0.0566      0.0253 (98.7604, 98.8596)  1.9600     15.6579 9.72e-05         Yes
       Precision    96.3320   0.0466   0.0208 (96.2912, 96.3728)       94.7940      0.0754      0.0337 (94.7279, 94.8601)  1.9600     31.5657 6.00e-06         Yes
          Recall    99.3160   0.0270   0.0121 (99.2923, 99.3397)       97.9000      0.0381      0.0170 (97.8666, 97.9334)  1.9600     49.8765 9.67e-07         Yes
        F1-score    99.0900   0.0158   0.0071 (99.0761, 99.1039)       98.2160      0.0594      0.0266 (98.1639, 98.2681)  1.9600     29.0367 8.37e-06         Yes
Dice coefficient    95.1080   0.0277   0.0124 (95.0837, 95.1323)       92.5500      0.0412      0.0184 (92.5139, 92.5861)  1.9600    279.

In [14]:
# ===========================================================
# STEP 5: Define BUSBRA External Validation Results
# ===========================================================

busbra_metrics = {

    "Accuracy": [97.88, 97.95, 97.89, 97.93, 97.91],

    "Precision": [97.90, 97.94, 97.88, 97.92, 97.91],

    "Recall": [97.87, 97.96, 97.90, 97.92, 97.91],

    "F1-score": [97.88, 97.95, 97.89, 97.93, 97.91],

    "Dice coefficient": [87.50, 88.10, 87.60, 87.90, 87.81],

    "MAE": [42.10, 42.85, 42.30, 42.60, 42.43],

    "MSE": [4.58e5, 4.62e5, 4.59e5, 4.61e5, 4.60e5],

    "RMSE": [678.0, 681.0, 679.0, 680.0, 680.0],

    "R2": [97.70, 97.90, 97.80, 97.85, 97.80]
}

print("BUSBRA external validation metrics loaded.")

BUSBRA external validation metrics loaded.


In [15]:
# ===========================================================
# STEP 6: BUSBRA Mean ± SD + 95% CI
# ===========================================================

busbra_results = []

for metric, values in busbra_metrics.items():

    stats_v = compute_statistics(values)

    busbra_results.append({

        "Metric": metric,
        "Mean": format_value(stats_v["Mean"]),
        "SD": format_value(stats_v["SD"]),
        "SE": format_value(stats_v["SE"]),
        "z-score": format_value(stats_v["z-score"]),
        "95% CI": format_ci(stats_v["CI Lower"], stats_v["CI Upper"])
    })

busbra_stats_df = pd.DataFrame(busbra_results)

print("\n=== BUSBRA External Validation Statistics ===")
print(busbra_stats_df.to_string(index=False))


=== BUSBRA External Validation Statistics ===
          Metric     Mean       SD       SE z-score               95% CI
        Accuracy  97.9120   0.0286   0.0128  1.9600   (97.8869, 97.9371)
       Precision  97.9100   0.0224   0.0100  1.9600   (97.8904, 97.9296)
          Recall  97.9120   0.0327   0.0146  1.9600   (97.8833, 97.9407)
        F1-score  97.9120   0.0286   0.0128  1.9600   (97.8869, 97.9371)
Dice coefficient  87.7820   0.2390   0.1069  1.9600   (87.5725, 87.9915)
             MAE  42.4560   0.2862   0.1280  1.9600   (42.2051, 42.7069)
             MSE 4.60e+05 1.58e+03 707.1068  1.9600 (4.59e+05, 4.61e+05)
            RMSE 679.6000   1.1402   0.5099  1.9600 (678.6006, 680.5994)
              R2  97.8100   0.0742   0.0332  1.9600   (97.7450, 97.8750)


In [16]:
# ===========================================================
# STEP 7: Save Formatted Statistical Results
# ===========================================================

import os

# -----------------------------------------------------------
# Define save directory
# -----------------------------------------------------------
save_dir = '/content/drive/MyDrive/Colab_Notebooks'

# Create directory if it does not exist
os.makedirs(save_dir, exist_ok=True)

# -----------------------------------------------------------
# Define save paths
# -----------------------------------------------------------
busi_save_path = os.path.join(
    save_dir,
    "BUSI_statistical_analysis_formatted.csv"
)

busbra_save_path = os.path.join(
    save_dir,
    "BUSBRA_external_validation_statistics_formatted.csv"
)

# -----------------------------------------------------------
# Save CSV files
# -----------------------------------------------------------
busi_stats_df.to_csv(
    busi_save_path,
    index=False
)

busbra_stats_df.to_csv(
    busbra_save_path,
    index=False
)

# -----------------------------------------------------------
# Display save paths
# -----------------------------------------------------------
print("\nBUSI formatted statistics saved at:")
print(busi_save_path)

print("\nBUSBRA formatted statistics saved at:")
print(busbra_save_path)


BUSI formatted statistics saved at:
/content/drive/MyDrive/Colab_Notebooks/BUSI_statistical_analysis_formatted.csv

BUSBRA formatted statistics saved at:
/content/drive/MyDrive/Colab_Notebooks/BUSBRA_external_validation_statistics_formatted.csv
